# Урок 18. Reinforcement Learning — обучение с подкреплением
**Библиотеки:** numpy

**Цель:** понять агента/среду/награды и обучить Q-learning агента в grid world.

## Простыми словами
RL — третий тип ML. Нет ни ответов (supervised), ни просто данных (unsupervised).
Есть **агент**, который ДЕЙСТВУЕТ в **среде** и получает **награды**. Как дрессировка щенка:
сделал хорошо → лакомство, плохо → ничего. Со временем щенок понимает, ЧТО приводит к лакомству.

Словарь: **State** — где агент сейчас. **Action** — что может сделать. **Reward** — очки за действие.
**Policy** — стратегия: какое действие выбирать в каждом state.

**Q-learning:** таблица Q[state, action] = «насколько хорошо сделать это действие здесь».
Агент гуляет, получает награды и обновляет таблицу:
> Q[s,a] += alpha * (reward + gamma * max(Q[s_новый]) − Q[s,a])

gamma — насколько ценим будущие награды. **Epsilon-greedy:** с вероятностью epsilon ходим
случайно (исследуем), иначе — по лучшему Q (используем знание).

In [ ]:
import numpy as np

# Grid world 4x4: старт (0,0), цель (3,3). Шаг = -1 очко, цель = +10.
SIZE = 4
ACTIONS = [(-1,0), (1,0), (0,-1), (0,1)]   # вверх, вниз, влево, вправо
GOAL = (3, 3)

rng = np.random.default_rng(0)
Q = np.zeros((SIZE, SIZE, 4))               # Q-таблица: 16 клеток x 4 действия
alpha, gamma, epsilon = 0.1, 0.9, 0.2

def step(state, a_idx):
    dr, dc = ACTIONS[a_idx]
    r, c = state[0] + dr, state[1] + dc
    r, c = max(0, min(SIZE-1, r)), max(0, min(SIZE-1, c))   # стены: остаёмся на месте
    reward = 10 if (r, c) == GOAL else -1
    return (r, c), reward

for episode in range(500):                  # 500 попыток дойти до цели
    s = (0, 0)
    while s != GOAL:
        if rng.uniform() < epsilon:                      # исследуем
            a = rng.integers(0, 4)
        else:                                            # используем знание
            a = Q[s[0], s[1]].argmax()
        s2, r = step(s, a)
        # Главная формула Q-learning:
        Q[s[0], s[1], a] += alpha * (r + gamma * Q[s2[0], s2[1]].max() - Q[s[0], s[1], a])
        s = s2

print("Обучение завершено. Ценность лучшего действия в каждой клетке:")
print(Q.max(axis=2).round(1))

In [ ]:
# Проверяем выученный маршрут (идём жадно по Q без случайности)
s, path = (0, 0), [(0, 0)]
arrows = ['↑', '↓', '←', '→']
route = []
while s != GOAL and len(path) < 20:
    a = Q[s[0], s[1]].argmax()
    route.append(arrows[a])
    s, _ = step(s, a)
    path.append(s)
print("Маршрут агента:", ' '.join(route))
print("Клетки:", path)
print("Шагов:", len(path) - 1, "(оптимум для 4x4 = 6)")

## Что произошло
Сначала агент тыкался случайно. Каждый удачный заход к цели «подкрашивал» Q-значения клеток
по дороге (награда просачивается назад благодаря gamma * max(Q[следующая клетка])).
Через 500 эпизодов агент идёт к цели за 6 шагов — оптимально. Никто не говорил ему маршрут —
он вывел его из наград. Так же (только с нейросетью вместо таблицы) учат играть в игры и управлять роботами.

## Типичные ошибки
1. epsilon=0 с самого начала — агент не исследует и застревает в первом найденном пути.
2. Забыть отрицательную награду за шаг — агенту незачем спешить, маршруты раздуваются.
3. gamma=0 — агент видит только сиюминутную награду и не планирует.

## Конспект 📓
RL: агент, среда, state, action, reward, policy. Q[s,a] = ценность действия в состоянии.
Обновление: Q += alpha*(r + gamma*maxQ(s') − Q). Epsilon-greedy: исследование vs использование.
Это база; в больших задачах Q-таблицу заменяет нейросеть (Deep Q-Network).

---
## Мини-задание 💪
Добавь «яму» в клетке (1,1): попадание даёт −10 и возврат на старт. Переобучи агента. Обходит ли он яму? Покажи новый маршрут.

## 5 проверочных вопросов ❓
1. Чем RL отличается от supervised и unsupervised?
2. Объясни state, action, reward на примере шахмат.
3. Что хранится в Q-таблице?
4. Зачем нужен epsilon и что будет при epsilon=0?
5. За что отвечает gamma?

## Домашнее задание 🏠
Задача 50 из practice_tasks.md. Поэкспериментируй с gamma=0.5 и gamma=0.99 — как меняется поведение?
